# Laboratoire : de la fonction `step` à l'agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro860/particle_gymnasium_sb3.ipynb)

Ce laboratoire construit, pas à pas, tout ce qu'il faut pour entraîner un agent d'apprentissage par renforcement sur un problème de commande que vous définissez vous-même.

- **Partie 0 — Les concepts, sans bibliothèque.** Un système en temps continu, sa version à pas de temps discret, et trois fonctions : reset, step et une politique. C'est tout ce qu'est un environnement.
- **Partie 1 — L'interface Gymnasium.** Le même environnement écrit selon le standard du domaine, entraîné avec Stable-Baselines3, puis les leçons pratiques : normaliser les actions, les observations et la récompense.
- **Partie 2 — Le même problème avec minilink.** Une courte démonstration, sans exercice : le problème déclaré une seule fois, résolu par apprentissage par renforcement natif, puis animé.

Les cellules marquées `# TODO: à compléter` sont à écrire ; les questions attendent une réponse de quelques lignes.

Cette page utilise [Gymnasium](https://gymnasium.farama.org/), [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) et la boîte à outils [minilink](https://github.com/alx87grd/minilink).

In [ ]:
# Local: dependencies already installed. Colab: clone minilink + path + RL libraries.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q gymnasium stable-baselines3 pygame")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pygame
from IPython.display import HTML
from matplotlib import animation

## Partie 0 — Les concepts, sans bibliothèque

En apprentissage par renforcement, un *agent* observe un *environnement*, agit, et reçoit une *récompense*.
Avant toute bibliothèque, voyons ce qu'est un environnement :

- un système dynamique $f$
- une fonction de coût $g$
- une distribution d'états initiaux $x_0$
- une règle de fin d'épisode

### Le problème

Une particule de masse $m$ se déplace sur un plan carré de 100 m de côté, dans un milieu visqueux de coefficient $b$.
Son état est sa position et sa vitesse, et des propulseurs appliquent une force bornée sur chaque axe :
$$x = [p_x,\; p_y,\; v_x,\; v_y], \qquad u = [F_x,\; F_y], \qquad |F_x|, |F_y| \le F_{\max}.$$
L'objectif est d'amener la particule au centre du plan, au repos ($\bar x = [50,\; 50,\; 0,\; 0]$), en forçant le moins possible :
$$g(x,u) = (x-\bar x)^T Q\,(x-\bar x) + u^T R\,u.$$
La particule ne doit pas sortir du plan.

In [ ]:
# Problem data
m = 10.0  # mass [kg]
b = 5.0  # viscous friction [N s/m]
F_MAX = 100.0  # force limit on each axis [N]
SIZE = 100.0  # side of the square plane [m]
V_MAX = 25.0  # speed bound [m/s] (the terminal speed is F_MAX / b = 20 m/s)

X_LOW = np.array([0.0, 0.0, -V_MAX, -V_MAX])  # state box: the plane and the speed bound
X_HIGH = np.array([SIZE, SIZE, V_MAX, V_MAX])
X_TARGET = np.array([SIZE / 2, SIZE / 2, 0.0, 0.0])  # center of the plane, at rest

# Cost weights
Q = np.diag([1.0, 1.0, 0.1, 0.1])
R = np.diag([0.01, 0.01])


def g(x, u):
    """Running cost g(x, u), per second."""
    e = x - X_TARGET
    return e @ Q @ e + u @ R @ u

### Exercice 0.1 — La dynamique en temps continu

Écrivez la deuxième loi de Newton sur chaque axe, avec le frottement visqueux $-b\,v$, et mettez le résultat sous la forme
$$\dot x = f(x, u).$$
La cellule de test applique la force maximale selon $x$ à la particule immobile au centre : vous devriez obtenir $\dot x = [0,\; 0,\; 10,\; 0]$.

In [ ]:
def f(x, u):
    """Continuous-time dynamics: returns dx/dt."""
    # TODO: à compléter
    raise NotImplementedError("à compléter")
    return dx


f(X_TARGET, np.array([F_MAX, 0.0]))

### Du temps continu à un pas de temps

Un agent ne décide pas en continu : il choisit une action, la maintient pendant une période $\Delta t$, puis observe le résultat.
Entre deux décisions, l'entrée est constante (bloqueur d'ordre zéro), et la méthode d'Euler donne le système à pas de temps discret :
$$x_{k+1} = x_k + f(x_k, u_k)\,\Delta t.$$


Le coût d'une période devient la récompense, avec la convention :
$$r_k = -g(x_k, u_k)\,\Delta t, \qquad J = \sum_k g(x_k,u_k)\,\Delta t = -\sum_k r_k.$$
Maximiser la somme des récompenses revient à minimiser $J$.

Enfin, l'épisode se termine si la particule sort du plan (ou dépasse la vitesse permise) : c'est un échec.
On facture alors une pénalité fixe, `PENALTY`, à la place de tout le coût qui restait à payer.
Nous prenons le pire coût possible d'un épisode complet, $g_{\max}\,T$ : ainsi, sortir n'est jamais moins cher que rester. Nous y reviendrons à la question 1.2.

In [ ]:
DT = 0.1  # control period [s]
T_EPISODE = 20.0  # episode duration [s]
N_STEPS = int(T_EPISODE / DT)  # 200 steps per episode

G_MAX = g(np.array([0.0, 0.0, V_MAX, V_MAX]), np.array([F_MAX, F_MAX]))  # worst running cost in the box
PENALTY = G_MAX * T_EPISODE  # price of leaving the box: the worst cost of a whole episode
PENALTY

### Exercice 0.2 — La fonction `step`

Écrivez `step(x, u)` comme une **fonction pure** : elle reçoit l'état et l'action, et retourne l'état suivant, la récompense et un indicateur de fin, sans rien mémoriser entre deux appels.
Découpez-la en quatre sous-étapes :

1. **Action admissible** : saturer $u$ à $\pm F_{\max}$ (`np.clip`) ;
2. **Récompense** : $r = -g(x, u)\,\Delta t$ ;
3. **Dynamique** : $x' = x + f(x,u)\,\Delta t$ ;
4. **Fin d'épisode** : `done` est vrai si $x'$ sort de la boîte `[X_LOW, X_HIGH]` ; dans ce cas, retranchez `PENALTY` de la récompense.

Test : depuis le centre, au repos, avec la force maximale selon $x$, vous devriez obtenir $x' = [50,\; 50,\; 1,\; 0]$, $r = -10$ et `done = False`.

In [ ]:
def step(x, u):
    """One control period, as a pure function: (x_k, u_k) -> (x_k+1, r_k, done)."""
    # TODO: à compléter
    raise NotImplementedError("à compléter")
    return x_next, r, done


step(X_TARGET, np.array([F_MAX, 0.0]))

### Exercice 0.3 — La fonction `reset`

L'état initial est aléatoire : position uniforme sur le plan, vitesse nulle.
Écrivez `reset(rng)`, où `rng` est un générateur aléatoire NumPy (`rng.uniform(a, b)` tire un nombre entre `a` et `b`).
Passer le générateur en argument rend l'expérience reproductible : même graine, mêmes états initiaux.

In [ ]:
def reset(rng):
    """Draw an initial state x0: uniform position on the plane, at rest."""
    # TODO: à compléter
    raise NotImplementedError("à compléter")
    return x0

for seed in range(10):
    print('New random x0:', reset(np.random.default_rng(seed)) )

### Voir l'environnement

`draw(x, u)` dessine le plan avec pygame : la cible en vert, la particule en bleu et la force appliquée en rouge.
pygame dessine sur une surface hors écran, ce qui fonctionne aussi sur Colab ; `show(frames)` assemble les images en une animation dans le carnet.
Ces deux fonctions sont fournies.

In [ ]:
PIXELS_PER_METER = 4


def draw(x, u):
    """Image of the plane (RGB array): target in green, particle in blue, force in red."""
    surface = pygame.Surface((int(SIZE * PIXELS_PER_METER), int(SIZE * PIXELS_PER_METER)))
    surface.fill((255, 255, 255))
    pygame.draw.rect(surface, (150, 150, 150), surface.get_rect(), 3)  # edges of the plane
    target = pygame.Vector2(X_TARGET[0], SIZE - X_TARGET[1]) * PIXELS_PER_METER  # y axis pointing up
    particle = pygame.Vector2(x[0], SIZE - x[1]) * PIXELS_PER_METER
    force = pygame.Vector2(u[0], -u[1]) / F_MAX * 15 * PIXELS_PER_METER  # 15 m long at full force
    pygame.draw.circle(surface, (0, 170, 0), target, 6)
    pygame.draw.line(surface, (220, 0, 0), particle, particle + force, 3)
    pygame.draw.circle(surface, (0, 0, 255), particle, 8)
    return pygame.surfarray.array3d(surface).transpose(1, 0, 2)


def show(frames):
    """Inline animation of a list of RGB images, one per control period."""
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.axis("off")
    image = ax.imshow(frames[0])
    anim = animation.FuncAnimation(
        fig, lambda k: image.set_data(frames[k]), frames=len(frames), interval=1000 * DT
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())

### La politique

Une **politique** est une fonction $u = \pi(x)$. La plus simple tire des forces au hasard.

In [ ]:
rng = np.random.default_rng(0)


def policy_random(x):
    return rng.uniform(-F_MAX, F_MAX, size=2)


for _ in range(10):
    print('Sample action of the policy:', policy_random( np.array([0.0, 0.0, 0.0, 0.0])) )

### La boucle agent–environnement


La boucle ci-dessous est tout ce que fait un environnement d'apprentissage par renforcement : `reset`, puis `step` jusqu'à la fin de l'épisode, en accumulant le coût $J$.

In [ ]:
x = reset(rng)
J, frames = 0.0, []

for k in range(N_STEPS):
    
    u = policy_random(x)
    frames.append(draw(x, u))

    ############################################################
    x, r, done = step(x, u)
    ############################################################

    J = J - r
    if done:
        break

print(f"{k + 1} steps, J = {J:.0f}")
show(frames)

### Optionnel, en local seulement : soyez l'agent

Sur votre ordinateur (pas sur Colab, qui n'a pas d'écran), mettez `PLAY_WITH_KEYBOARD = True` : une fenêtre pygame s'ouvre et les flèches du clavier deviennent la politique.
Amenez la particule au centre et arrêtez-l'y, puis comparez votre $J$ à celui des politiques de l'exercice suivant.
La touche `q` ou la fermeture de la fenêtre arrête l'épisode.

In [ ]:
PLAY_WITH_KEYBOARD = False

if PLAY_WITH_KEYBOARD:
    pygame.display.init()
    screen = pygame.display.set_mode((int(SIZE * PIXELS_PER_METER), int(SIZE * PIXELS_PER_METER)))
    clock = pygame.time.Clock()
    x, J, done, k = reset(rng), 0.0, False, 0
    while not done and k < N_STEPS:
        closed = any(event.type == pygame.QUIT for event in pygame.event.get())
        keys = pygame.key.get_pressed()
        if closed or keys[pygame.K_q]:
            break
        u = F_MAX * np.array(
            [keys[pygame.K_RIGHT] - keys[pygame.K_LEFT], keys[pygame.K_UP] - keys[pygame.K_DOWN]],
            dtype=float,
        )
        screen.blit(pygame.surfarray.make_surface(draw(x, u).transpose(1, 0, 2)), (0, 0))
        pygame.display.flip()
        x, r, done = step(x, u)
        J, k = J - r, k + 1
        clock.tick(1 / DT)  # real time
    pygame.display.quit()
    print(f"{k} steps, J = {J:.0f}")

### Évaluation d'une politique

On fournit ici une fonction qui évalue le coût moyen d'une politique :

In [ ]:
def evaluate(policy, n_episodes=20):
    """Mean cost J of a policy over n_episodes fixed initial states (seeds 0, 1, ...)."""
    costs = []
    for i in range(n_episodes):
        x = reset(np.random.default_rng(i))
        J = 0.0
        for k in range(N_STEPS):
            x, r, done = step(x, policy(x))
            J = J - r
            if done:
                break
        costs.append(J)
    return np.mean(costs)


def policy_zero(x):
    return np.zeros(2)


print(f"zero policy: J = {evaluate(policy_zero):.0f}")
print(f"random policy: J = {evaluate(policy_random):.0f}")

Notez le coût moyen des politiques 'ne rien faire' et 'actions aléatoires'. Idéalement on devrait être capable de concevoir une politique plus performante!!!

### Exercice 0.4 — Une politique écrite à la main

Écrivez une politique proportionnelle-dérivée saturée, où $p$ est la position et $v$ la vitesse :
$$u = \mathrm{sat}\big(K_p\,(\bar p - p) - K_d\,v\big).$$
Réglez $K_p$ et $K_d$ pour minimiser le coût moyen $J$ sur 20 états initiaux, calculé par `evaluate` (fournie), et comparez à la politique nulle $u = 0$.

In [ ]:
# TODO: à compléter
KP, KD = 1.0, 1.0  # tune these gains


def policy_pd(x):
    # TODO: à compléter
    raise NotImplementedError("à compléter")
    return np.clip(u, -F_MAX, F_MAX)


print(f"PD policy:   J = {evaluate(policy_pd):.0f}")

In [ ]:
x = reset(np.random.default_rng(0))
frames = []
for k in range(N_STEPS):
    u = policy_pd(x)
    frames.append(draw(x, u))
    x, r, done = step(x, u)
    if done:
        break
show(frames)

### À retenir

Un environnement, c'est `reset` (la distribution de $x_0$) et `step` ($f$, $g$, $\Delta t$ et la règle de fin) ; un agent, c'est une politique $\pi$.
L'apprentissage par renforcement cherche $\pi$ en n'appelant que `reset` et `step` : il ne voit jamais $f$ ni $g$, seulement des transitions $(x, u, r, x')$.
C'est la différence avec l'itération de valeurs, qui utilise $f$ et $g$ explicitement.

## Partie 1 — L'interface Gymnasium

[Gymnasium](https://gymnasium.farama.org/) standardise la façon d'écrire `reset` et `step`.
Presque toutes les bibliothèques d'apprentissage par renforcement s'entraînent contre cette interface : écrivez votre environnement une fois, et n'importe quel algorithme peut l'utiliser.
Survolez la page [Basic Usage](https://gymnasium.farama.org/introduction/basic_usage/) avant de continuer.

| Partie 0 | Gymnasium | |
|---|---|---|
| `x = reset(rng)` | `y, info = env.reset(seed=...)` | tirage de $x_0$ avec le générateur `self.np_random` |
| `x, r, done = step(x, u)` | `y, r, terminated, truncated, info = env.step(u)` | une période $\Delta t$ |
| l'état $x$ passe d'un appel à l'autre | l'état est **dans l'objet** (`self.x`) | `step` n'est plus une fonction pure |
| `X_LOW`, `X_HIGH` | `observation_space` | l'ensemble des observations $y$ |
| `F_MAX` | `action_space` | les actions admissibles |
| `draw(x, u)` | `render()` | une image de l'état courant |

Deux différences de fond :

- **L'observation.** L'agent reçoit $y = h(x)$, pas forcément l'état complet. Ici, $y = x$.
- **Deux façons de finir un épisode.** *terminated* : l'épisode est vraiment fini, il ne reste rien à payer (but atteint, échec, fin d'un horizon fini). *truncated* : on arrête de simuler, mais le futur avait encore un coût (limite de temps). Un algorithme qui apprend un coût-à-venir estime la suite d'un épisode *truncated* par sa propre valeur de l'état final ; après *terminated*, la suite vaut zéro.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env

### Exercice 1.1 — Votre environnement Gymnasium

Complétez `ParticleEnv` en réutilisant `f`, `g` et `draw` de la partie 0 :

- `__init__` : `observation_space` et `action_space` sont des [`spaces.Box`](https://gymnasium.farama.org/api/spaces/fundamental/#box) aux bornes de l'état et de la force, de type `np.float32` ;
- `reset` : la même distribution qu'à la partie 0, tirée avec le générateur `self.np_random` ; mémorisez l'état dans `self.x` (et `self.u = np.zeros(2)` pour l'affichage) ; retournez l'observation en `np.float32` ;
- `step` : les quatre sous-étapes de la partie 0, en mettant à jour `self.x` et `self.u`. Sortir de la boîte est *terminated* ; la limite de temps sera ajoutée par Gymnasium à l'enregistrement, donc `truncated` reste `False` ici. La dernière observation d'un épisode raté peut sortir de la boîte : ramenez-la dans `observation_space` avec `np.clip`.

Le [tutoriel officiel](https://gymnasium.farama.org/introduction/create_custom_env/) montre un autre exemple complet.

In [ ]:
class ParticleEnv(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": int(1 / DT)}

    def __init__(self, render_mode=None):
        # TODO: à compléter
        raise NotImplementedError("à compléter")
        self.render_mode = render_mode

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)  # seeds self.np_random
        # TODO: à compléter
        raise NotImplementedError("à compléter")
        return y, {}

    def step(self, action):
        # TODO: à compléter
        raise NotImplementedError("à compléter")
        return y, float(r), terminated, truncated, {}

    def render(self):
        return draw(self.x, self.u)

In [ ]:
check_env(ParticleEnv(render_mode="rgb_array"))

`check_env` vérifie le contrat : types, bornes, graines, `render`.
Notez son avertissement sur l'espace d'actions ; nous y reviendrons.

Pour utiliser `gym.make`, on enregistre l'environnement.
L'argument `max_episode_steps` l'enveloppe dans un [`TimeLimit`](https://gymnasium.farama.org/api/wrappers/misc_wrappers/#gymnasium.wrappers.TimeLimit), qui déclare *truncated* après 200 pas.
C'est votre première *enveloppe* (*wrapper*) : l'affichage de `env` montre la pile complète.

In [ ]:
gym.register(id="Particle-v0", entry_point=ParticleEnv, max_episode_steps=N_STEPS)
env = gym.make("Particle-v0", render_mode="rgb_array")
env

### Question 1.2 — *terminated* ou *truncated* ?

1. Pourquoi la sortie de la boîte est-elle *terminated*, alors que la limite de 20 s est *truncated* ?
2. `PENALTY` remplace tout le coût qui restait à payer. Que risque-t-on si elle est trop petite, par exemple 1 000 ? Prédisez le comportement appris ; l'exercice 1.5 permet de vérifier.

*Votre réponse :*

### Le même problème, la même politique

L'interface a changé, pas le problème : votre politique PD, passée par la boucle Gymnasium, donne le même $J$ qu'à la partie 0, car les mêmes graines donnent les mêmes états initiaux.

In [ ]:
def evaluate_env(env, policy, n_episodes=20):
    """Mean cost J = -sum(r) over n_episodes, through the Gymnasium loop (seeds 0, 1, ...)."""
    costs = []
    for i in range(n_episodes):
        y, info = env.reset(seed=i)
        J, terminated, truncated = 0.0, False, False
        while not (terminated or truncated):
            y, r, terminated, truncated, info = env.step(policy(y))
            J = J - r
        costs.append(J)
    return np.mean(costs)


print(f"PD, part 0 loop:    J = {evaluate(policy_pd):.0f}")
print(f"PD, Gymnasium loop: J = {evaluate_env(env, policy_pd):.0f}")

### Entraîner un agent avec Stable-Baselines3

[Stable-Baselines3](https://stable-baselines3.readthedocs.io/) (SB3) regroupe des implémentations de référence d'algorithmes d'apprentissage par renforcement ; une fois l'environnement écrit, l'entraînement tient en deux lignes.
L'algorithme est PPO : une politique neuronale $\pi_\theta(y)$, améliorée à partir de trajectoires simulées.
C'est une forme d'itération de politique où l'évaluation (le *critique*) et l'amélioration (l'*acteur*) s'apprennent à partir d'échantillons ; nous l'ouvrirons plus tard dans le cours.

`verbose=1` affiche `ep_rew_mean`, le retour moyen des derniers épisodes, c'est-à-dire $-J$ en cours d'apprentissage.
Comptez 30 s en local et 1 à 2 min sur Colab pour 100 000 pas.

In [ ]:
from stable_baselines3 import PPO

model = PPO("MlpPolicy", env, seed=0, verbose=1)
model.learn(total_timesteps=100_000)

### Politique apprise

Après l'entraînement de notre réseau de neurones, appelé 'model' (ici, `model` est le réseau de neurones, pas le modèle dynamique), on utilise sa prédiction la plus probable comme politique :

In [ ]:
def policy_ppo(y):
    """The trained network's mean action (no exploration noise); reads the current `model`."""
    return model.predict(y, deterministic=True)[0]


print(f"zero policy: J = {evaluate(policy_zero):.0f}")
print(f"PD policy:   J = {evaluate(policy_pd):.0f}")
print(f"PPO, raw:    J = {evaluate_env(env, policy_ppo):.0f}")

In [ ]:
y, info = env.reset(seed=0)
frames, terminated, truncated = [], False, False
while not (terminated or truncated):
    frames.append(env.render())
    y, r, terminated, truncated, info = env.step(policy_ppo(y))
show(frames)

Oooonh… y est pas bon.
Bienvenue en apprentissage par renforcement : ça marche rarement du premier coup, même sur un problème « simple ».
L'agent ne fait guère mieux que la politique nulle.

### Pourquoi ? Les échelles

Les réseaux de neurones, et PPO en particulier, supposent des grandeurs de l'ordre de 1 :

- **Les actions.** La politique de PPO est une gaussienne centrée sur la sortie du réseau, d'écart type initial 1. Sur un espace de $\pm 100$ N, l'agent explore donc avec des forces d'environ 1 N : il bouge à peine, et découvre très lentement qu'il faut pousser fort. C'est l'avertissement de `check_env`.
- **Les observations.** Les positions valent jusqu'à 100 : les neurones `tanh` de la première couche saturent, et le réseau distingue mal les états.
- **La récompense.** $r$ atteint $-500$ par pas et les retours $-10^4$ : le critique doit prédire des valeurs énormes, ses gradients dominent ceux de l'acteur, et l'écrêtage du gradient de SB3 (une norme maximale commune aux deux réseaux) freine alors tout l'apprentissage.

D'autres [conseils pratiques](https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html) sont dans la documentation de SB3.

On pourrait corriger l'environnement lui-même ; on va plutôt l'**envelopper**, sans y toucher.
Une enveloppe Gymnasium transforme une seule chose : l'action (`gym.ActionWrapper`, méthode `action`), l'observation (`gym.ObservationWrapper`, méthode `observation`) ou la récompense (`gym.RewardWrapper`, méthode `reward`).
Vous en utilisez déjà deux : `TimeLimit`, et `PassiveEnvChecker`, qui vérifie les sorties de l'environnement ; voyez la [liste des enveloppes fournies](https://gymnasium.farama.org/api/wrappers/table/).

### Exercice 1.3 — Trois enveloppes

1. `NormalizeAction` : l'agent choisit $a \in [-1, 1]^2$, et l'enveloppe le convertit en force de façon affine, $a = -1 \mapsto$ `low` et $a = 1 \mapsto$ `high` de l'espace d'origine.
2. `NormalizeObservation` : ramène $y$ dans $[-1, 1]$, avec les bornes de l'espace d'observation d'origine.
3. `ScaleReward` : multiplie la récompense par une constante `scale`.

Dans les deux premiers cas, redéfinissez aussi l'espace que voit l'agent (`self.action_space` ou `self.observation_space`).
L'espace d'origine reste accessible par `self.env`.

In [ ]:
class NormalizeAction(gym.ActionWrapper):
    """The agent acts in [-1, 1]; the environment receives forces in [low, high]."""

    def __init__(self, env):
        super().__init__(env)
        # TODO: à compléter
        raise NotImplementedError("à compléter")

    def action(self, a):
        # TODO: à compléter
        raise NotImplementedError("à compléter")
        return u


class NormalizeObservation(gym.ObservationWrapper):
    """The agent observes y scaled to [-1, 1] with the bounds of the observation space."""

    def __init__(self, env):
        super().__init__(env)
        # TODO: à compléter
        raise NotImplementedError("à compléter")

    def observation(self, y):
        # TODO: à compléter
        raise NotImplementedError("à compléter")
        return y


class ScaleReward(gym.RewardWrapper):
    """The agent receives scale * r."""

    def __init__(self, env, scale):
        super().__init__(env)
        self.scale = scale

    def reward(self, r):
        # TODO: à compléter
        raise NotImplementedError("à compléter")
        return r

Test : avec $a = [1,\; -1]$, la force appliquée doit être $[100,\; -100]$ N, l'observation doit être dans $[-1, 1]$, et la récompense mille fois plus petite qu'avant.

In [ ]:
wrapped = ScaleReward(NormalizeObservation(NormalizeAction(gym.make("Particle-v0", render_mode="rgb_array"))), scale=1e-3)
y, info = wrapped.reset(seed=0)
y, r, terminated, truncated, info = wrapped.step(np.array([1.0, -1.0]))
print("agent sees:", wrapped.action_space, wrapped.observation_space)
print("force applied:", wrapped.unwrapped.u)

Gymnasium fournit des équivalents : `gym.wrappers.RescaleAction(env, -1.0, 1.0)`, `gym.wrappers.TransformObservation` et `gym.wrappers.TransformReward(env, lambda r: 1e-3 * r)`.
Attention : `gym.wrappers.NormalizeObservation` et `gym.wrappers.NormalizeReward` ne font pas la même chose que nos classes ; ils normalisent avec des moyennes et des écarts types estimés au fil de l'entraînement, ce qui est utile quand on ne connaît pas les bornes.

### PPO sur l'environnement normalisé

In [ ]:
model2 = PPO("MlpPolicy", wrapped, seed=0, verbose=1)
model2.learn(total_timesteps=100_000)

In [ ]:
def policy_ppo2(y):
    """The trained network's mean action (no exploration noise); reads `model2`."""
    return model2.predict(y, deterministic=True)[0]


print(f"zero policy: J = {evaluate(policy_zero):.0f}")
print(f"PD policy:   J = {evaluate(policy_pd):.0f}")
print(f"PPO, raw:    J = {evaluate_env(env, policy_ppo):.0f}")
print(f"PPO, scaled: J = {evaluate_env(wrapped, policy_ppo2) / wrapped.scale:.0f}")  # undo the scaling

Fonctionne mieux maintenant!

In [ ]:
y, info = wrapped.reset(seed=0)
frames, terminated, truncated = [], False, False
while not (terminated or truncated):
    frames.append(wrapped.render())
    y, r, terminated, truncated, info = wrapped.step(policy_ppo2(y))
show(frames)

**Poursuivre un entraînement.** Si l'agent n'a pas fini d'apprendre, inutile de repartir de zéro :
`model2.learn(total_timesteps=50_000, reset_num_timesteps=False)` poursuit l'entraînement du même réseau, et le compteur de pas (avec `ep_rew_mean`) reprend là où il s'était arrêté.

### Exercice 1.4 — L'expérience

Quelle correction compte le plus ? Comparons quatre configurations, toutes avec les actions normalisées, un prérequis de PPO.
La cellule suivante entraîne PPO (100 000 pas, graine 0) dans les configurations A, B et C ; la configuration D est l'agent `model2`, déjà entraîné plus haut.

| | actions | observations | récompense $\times 10^{-3}$ |
|---|---|---|---|
| A | ✓ | | |
| B | ✓ | ✓ | |
| C | ✓ | | ✓ |
| D | ✓ | ✓ | ✓ |

Remarquez comment chaque agent est évalué : `evaluate_env` sur les mêmes enveloppes d'action et d'observation qu'à l'entraînement, mais **sans** la mise à l'échelle de la récompense (pour D, on divise par `wrapped.scale`, ce qui revient au même).
On compare ainsi le vrai coût $J$ de chaque agent ; des récompenses mises à l'échelle différemment ne seraient pas comparables.
Exécutez-la (comptez 1 min 30 en local et 4 min sur Colab), puis répondez aux questions.

In [ ]:
REWARD_SCALE = wrapped.scale  # the scale model2 was trained with (1e-3)
results = {}
for name, norm_obs, scale_rew in [("A", False, False), ("B", True, False), ("C", False, True)]:
    eval_env = NormalizeAction(gym.make("Particle-v0", render_mode="rgb_array"))
    if norm_obs:
        eval_env = NormalizeObservation(eval_env)
    train_env = ScaleReward(eval_env, REWARD_SCALE) if scale_rew else eval_env
    model = PPO("MlpPolicy", train_env, seed=0)
    model.learn(total_timesteps=100_000)
    results[name] = evaluate_env(eval_env, policy_ppo)  # true cost J: same wrappers, no reward scaling
    print(f"{name}: J = {results[name]:.0f}")

results["D"] = evaluate_env(wrapped, policy_ppo2) / wrapped.scale  # D is model2, already trained
print(f"D: J = {results['D']:.0f}")

In [ ]:
names = ["A", "B", "C", "D"]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(names, [results[n] for n in names])
ax.axhline(evaluate(policy_pd), color="k", linestyle="--", label="PD")
ax.axhline(evaluate(policy_zero), color="gray", linestyle=":", label="u = 0")
ax.set_ylabel("mean cost J")
ax.legend()
plt.show()

**Questions d'analyse**

1. Classez les configurations. Quelle correction apporte le plus, en plus de la normalisation des actions ?
2. Ces résultats viennent d'une seule graine. Peut-on conclure que B est meilleure que C ? Que faudrait-il faire avant de conclure ? (Si le temps le permet, relancez B et C avec `seed=1`.)
3. Le meilleur agent bat-il votre PD ? Comparez le nombre de pas de simulation qu'il lui a fallu et le temps qu'il vous a fallu pour régler le PD. Dans quelles situations l'apprentissage vaut-il la peine ?

*Votre réponse :*

### Exercice 1.5 — Une pénalité trop faible

À la question 1.2, vous avez prédit ce qui arrive si `PENALTY` est trop petite ; vérifiez-le.
La cellule ci-dessous met `PENALTY = 1000` (la méthode `step` lit la variable globale à chaque appel).
Entraînez PPO dans la configuration D (graine 0), puis comptez, sur les 20 épisodes d'évaluation (graines 0 à 19), ceux qui finissent hors du plan, et leur durée moyenne.
Visualisez un épisode.
La cellule suivante remet la pénalité à sa valeur.

In [ ]:
PENALTY = 1000.0  # too small; step() reads the global PENALTY at each call
# TODO: à compléter
raise NotImplementedError("à compléter")

In [ ]:
PENALTY = G_MAX * T_EPISODE  # back to the worst cost of a whole episode

### Exercice 1.6 — Les hyperparamètres

La [documentation de PPO](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html) liste ses hyperparamètres.
Trois se confondent facilement :

- `n_steps` : le nombre de pas simulés entre deux mises à jour de la politique. C'est le jeu de données de chaque itération, souvent appelé *rollout*.
- `batch_size` : le nombre d'échantillons de ce rollout utilisés pour chaque pas de gradient, comme en apprentissage supervisé.
- `n_epochs` : le nombre de passages sur tout le rollout à chaque itération, comme en apprentissage supervisé.

La [forme des réseaux](https://stable-baselines3.readthedocs.io/en/master/guide/custom_policy.html) se choisit avec `policy_kwargs`, par exemple `PPO("MlpPolicy", env, policy_kwargs=dict(net_arch=[128, 128], activation_fn=torch.nn.ReLU))`.
Essayez au moins une variante sur la configuration D (graine 0) et comparez son $J$ à celui de l'expérience.
Vous aurez besoin de ces réglages pour le projet.

In [ ]:
import torch

# TODO: à compléter
raise NotImplementedError("à compléter")

### Exercice 1.7 — Un autre algorithme : SAC

Entraînez un agent [SAC](https://stable-baselines3.readthedocs.io/en/master/modules/sac.html) sur la configuration D, évaluez-le et visualisez-le.
SAC réutilise ses anciennes transitions (*replay buffer*) : il demande moins de pas de simulation, mais chaque pas coûte plus de calcul (comptez 5 min pour 30 000 pas).
Attention : SAC met la récompense en balance avec un bonus d'entropie, et l'échelle de récompense qui convient à PPO ne lui convient pas forcément ; essayez-en quelques-unes.

In [ ]:
from stable_baselines3 import SAC

# TODO: à compléter
raise NotImplementedError("à compléter")

### Remarques importantes

La principale difficulté de l'apprentissage par renforcement : il n'y a pas de jeu de données fixe.
**L'agent génère ses propres données pendant l'entraînement.**
Chaque changement de paramètre, d'échelle ou de graine change les données qu'il voit et qu'il crée : s'il explore une région inutile ou s'il reste mauvais, il continue de produire des données inutiles et n'apprend pas.
D'où la sensibilité aux échelles, aux graines et aux hyperparamètres observée plus haut.

Respectez aussi l'hypothèse de Markov : l'observation doit résumer tout ce qui compte pour la suite.
Si la même action dans la même observation mène à des récompenses ou à des états différents selon une variable cachée (une vitesse non observée, un compteur de temps interne…), l'agent ne voit pas de signal cohérent, et l'apprentissage échoue.

## Partie 2 — Le même problème avec minilink (démonstration)

Jusqu'ici, le problème a été écrit trois fois : des fonctions à la partie 0, une classe à la partie 1, puis des enveloppes pour les échelles.
Chaque version mélange la **définition du problème** ($f$, $g$, la loi de $x_0$, la règle de fin) et **l'interface** qui le présente à un algorithme.

[minilink](https://github.com/alx87grd/minilink), la boîte à outils du cours, sépare les deux.
Le problème est déclaré une seule fois, comme un `StochasticPlanningProblem` : le système ($f$), le coût ($g$), la boîte $X$ et le prix pour en sortir (notre `PENALTY`), et la distribution de $x_0$.
À partir de cet objet, plusieurs outils s'appliquent sans réécrire le problème.
Cette partie est une démonstration : il n'y a rien à compléter.

In [ ]:
from minilink import (
    DynamicSystem,
    QuadraticCost,
    ReinforcementLearningPlanner,
    StochasticPlanningProblem,
    Uniform,
)
from minilink.graphical.catalog import Circle, Line, Point, point_pose
from minilink.interfaces.gymnasium import Sys2Gym

In [ ]:
# The particle is linear: dx/dt = A x + B sat(u)
A = np.array([[0, 0, 1, 0], [0, 0, 0, 1], [0, 0, -b / m, 0], [0, 0, 0, -b / m]])
B = np.array([[0, 0], [0, 0], [1 / m, 0], [0, 1 / m]])


class Particle(DynamicSystem):
    def __init__(self):
        super().__init__(n=4, input_dim=2, output_dim=4, expose_state=True)
        self.name = "Particle"
        self.state.labels = ["px", "py", "vx", "vy"]
        self.state.lower_bound = X_LOW.copy()
        self.state.upper_bound = X_HIGH.copy()
        self.inputs["u"].labels = ["Fx", "Fy"]
        self.inputs["u"].lower_bound = np.array([-F_MAX, -F_MAX])
        self.inputs["u"].upper_bound = np.array([F_MAX, F_MAX])
        self.skin = lambda sys: {  # shapes drawn by animate(), one list per frame
            "world": [
                Line([[0, 0], [SIZE, 0], [SIZE, SIZE], [0, SIZE], [0, 0]], color="gray"),
                Point((X_TARGET[0], X_TARGET[1], 0.0), color="green", size=8),
            ],
            "particle": [Circle(radius=2.0, color="blue", fill=True)],
        }

    def f(self, x, u, t=0.0, params=None):
        return A @ x + B @ u.clip(-F_MAX, F_MAX)  # the f of part 0, force saturated

    def tf(self, x, u, t=0.0, params=None):
        return {"particle": point_pose(x[:2])}  # pose of the particle frame


plant = Particle()

In [ ]:
cost = QuadraticCost.from_system(plant, xbar=X_TARGET, Q=Q, R=R)  # the g of part 0

problem = StochasticPlanningProblem(
    plant,
    cost=cost,
    tf=np.inf,
    X=plant.state.box,  # leaving the box is a failure...
    infeasible_cost=PENALTY,  # ...charged the same price
    x0_distribution=Uniform([0.0, 0.0, 0.0, 0.0], [SIZE, SIZE, 0.0, 0.0]),  # the x0 of part 0
)

**Une ligne pour Gymnasium.** `Sys2Gym.from_problem` produit l'environnement Gymnasium de ce problème — l'équivalent de votre `ParticleEnv`, prêt pour SB3.

In [ ]:
gym_env = Sys2Gym.from_problem(problem, dt=DT, episode_length=T_EPISODE, integrator="euler")
check_env(gym_env)
gym_env.observation_space, gym_env.action_space

**Apprentissage par renforcement natif.** `ReinforcementLearningPlanner` entraîne PPO sans la boucle Python de Gymnasium : il compile $f$ et $g$ avec JAX et simule des dizaines d'épisodes en parallèle.
Il applique d'office deux leçons de la partie 1 : la politique agit dans $[-1, 1]$, converti vers les bornes de la force, et elle observe l'état normalisé par la boîte $X$.
Il intègre $f$ avec Runge-Kutta 4 plutôt qu'avec Euler.
Comparez la durée affichée aux 100 000 pas de SB3.

In [ ]:
rl = ReinforcementLearningPlanner(problem, dt=DT, episode_length=T_EPISODE, gamma=0.99, verbose=False)
sol_ppo = rl.solve(timesteps=500_000)

In [ ]:
print(sol_ppo.solver)
ctl = sol_ppo.policy
ctl.plot_control_law()

In [ ]:
plant.x0 = np.array([10.0, 90.0, 0.0, 0.0])
closed_loop = ctl @ plant
traj = closed_loop.compute_trajectory(tf=T_EPISODE, dt=DT, verbose=False)

In [ ]:
closed_loop.animate(traj)

### Bilan

| | Partie 0 | Partie 1 : Gymnasium + SB3 | Partie 2 : minilink |
|---|---|---|---|
| Le problème | des fonctions | une classe et des enveloppes | un objet déclaré une fois |
| Ce que voit l'algorithme | — | `reset` et `step` : une boîte noire | $f$ et $g$ explicitement |
| Ce qu'on en fait | simuler, comprendre | n'importe quelle bibliothèque d'apprentissage | apprentissage compilé, LQR, itération de valeurs, évaluation |
| Quand s'en servir | pour apprendre | simulateur existant, jeu, problème sans modèle | problème de commande dont on connaît le modèle |

Ici, l'apprentissage par renforcement retrouve à peu près la performance d'un PD réglé à la main : il n'en vaut la peine que lorsque le problème sort des hypothèses des méthodes à base de modèle — dynamique non linéaire ou inconnue, contraintes, observations riches.
Pour votre projet : si votre problème s'écrit $\dot x = f(x, u)$ avec un coût $g$, déclarez-le une fois et essayez plusieurs méthodes ; sinon, écrivez un environnement Gymnasium comme à la partie 1.